# Timeline Triage Notebook

Interactive analysis of a Hayabusa `dfir-timeline` CSV, as a lightweight replacement for
Timesketch for the common case (see repo methodology). Intended input is
**`001-hayabusa-timeline.csv`**, produced by `evtx/evtx-triage.sh` (Hayabusa 4.0.0
`dfir-timeline`, CSV/standard profile).

Run this notebook from the repo venv:

```bash
source venv-setup/venv/bin/activate
jupyter lab notebooks/timeline-template.ipynb
```

**Design notes (measured against a real Hayabusa 4.0.0 run, not assumed):**

- `Level` values are the *abbreviated* Hayabusa strings `crit`/`high`/`med`/`low`/`info` —
  not `"critical"`/`"medium"`/`"informational"`. The level-threshold filter below uses an
  explicit ordinal map over these exact strings.
- `Channel` values are also abbreviated: `Sec`/`Sys`/`Sysmon`.
- `Timestamp` carries the *local* timezone offset of the machine that ran Hayabusa (e.g.
  `2019-04-27 23:04:32.373 +02:00`), not UTC — the same EVTX produces different offsets on
  different hosts. It is parsed with `format="mixed", utc=True` below.
- `Details` / `ExtraFieldInfo` use `¦` (U+00A6 BROKEN BAR) as an internal field separator,
  not a pipe character — relevant if you post-process those columns beyond the free-text
  search used here.
- Visualisation uses `msticpy.vis.timeline.display_timeline` called directly. This needs no
  `msticpyconfig.yaml` and emits no warning; the alternative (`init_notebook()` +
  `df.mp_plot.timeline()`) logs a `Could not find msticpyconfig.yaml` warning on every run
  and is not used here.

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
# CSV_PATH: path to the Hayabusa dfir-timeline CSV (001-hayabusa-timeline.csv) produced by
# evtx/evtx-triage.sh. Overridable via SOC_TIMELINE_CSV so this notebook can point at any
# triage run without editing the notebook itself (no hardcoded absolute paths).
CSV_PATH = Path(
    os.environ.get(
        "SOC_TIMELINE_CSV",
        "../evtx/output_<evtx_dir_name>_<timestamp>/001-hayabusa-timeline.csv",
    )
)
CSV_PATH

In [ ]:
df = pd.read_csv(CSV_PATH)

# Hayabusa 4.0.0 dfir-timeline writes Timestamp with a per-host local timezone offset
# (e.g. "2019-04-27 23:04:32.373 +02:00"), not UTC. format="mixed" tolerates the mixed
# offsets/precision across rows; utc=True normalises everything onto one tz-aware axis
# (verified to produce dtype datetime64[ns, UTC] on real Hayabusa output).
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="mixed", utc=True)

print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Timestamp dtype: {df['Timestamp'].dtype}")
df.head()

## Overview

Counts per `Level` / `Channel` / `Computer`, top `EventID`s, and the overall time range.

`require_columns()` is checked first and raises a clear, actionable error (naming the
missing column(s) and the columns actually present) rather than letting a later cell fail
with a bare `KeyError` — Hayabusa's CSV schema already changed once at v4.0.0 (the earlier
`csv-timeline`/`json-timeline` subcommands were merged into `dfir-timeline`), so this is a
real, not hypothetical, risk on a future Hayabusa upgrade.

In [ ]:
REQUIRED_COLUMNS: tuple[str, ...] = (
    "Timestamp", "RuleTitle", "Level", "Computer", "Channel",
    "EventID", "RecordID", "Details", "ExtraFieldInfo", "RuleID",
)


def require_columns(data: pd.DataFrame, columns: tuple[str, ...] = REQUIRED_COLUMNS) -> None:
    '''Raise a clear KeyError naming any missing columns instead of a bare KeyError downstream.

    Hayabusa's dfir-timeline CSV schema already changed once at v4.0.0 (csv-timeline and
    json-timeline were merged into dfir-timeline) -- a future version bump could rename or
    drop columns silently, so this check runs immediately after load.
    '''
    missing = [c for c in columns if c not in data.columns]
    if missing:
        raise KeyError(
            f"hayabusa timeline CSV is missing expected column(s): {missing}. "
            f"Present columns: {list(data.columns)}. Hayabusa's dfir-timeline schema may "
            f"have changed -- check the Hayabusa version and update REQUIRED_COLUMNS."
        )


require_columns(df)
print("All required columns present.")

In [ ]:
# Counts per Level / Channel / Computer, top EventIDs, overall time range.
level_counts = df["Level"].value_counts()
channel_counts = df["Channel"].value_counts()
computer_counts = df["Computer"].value_counts()
top_event_ids = df["EventID"].value_counts().head(10)
time_min, time_max = df["Timestamp"].min(), df["Timestamp"].max()

print(f"Time range: {time_min} -> {time_max}")
print(f"Level counts sum to len(df) ({len(df)}): {level_counts.sum() == len(df)}")
print(f"Channel counts sum to len(df) ({len(df)}): {channel_counts.sum() == len(df)}")

level_counts

In [ ]:
channel_counts

In [ ]:
computer_counts

In [ ]:
top_event_ids

## Parametric filters

Each filter rebuilds from the base `df` (not from a previously filtered frame) and returns
a `.copy()`. Edit the parameter variable in each cell and re-run just that cell.

In [ ]:
# Filter: Level >= threshold, using an explicit ordinal map over Hayabusa's abbreviated
# Level strings (crit/high/med/low/info). Comparing against "medium"/"critical" (the full
# words) would silently match nothing -- Hayabusa 4.0.0 never writes those.
LEVEL_ORDER: dict[str, int] = {"info": 0, "low": 1, "med": 2, "high": 3, "crit": 4}


def filter_by_level(data: pd.DataFrame, min_level: str) -> pd.DataFrame:
    '''Return rows with Level >= min_level per LEVEL_ORDER (ordinal, not lexical).'''
    if min_level not in LEVEL_ORDER:
        raise ValueError(f"min_level must be one of {sorted(LEVEL_ORDER)}, got {min_level!r}")
    threshold = LEVEL_ORDER[min_level]
    return data[data["Level"].map(LEVEL_ORDER) >= threshold].copy()


MIN_LEVEL = "high"  # one of: info, low, med, high, crit
filtered_by_level = filter_by_level(df, MIN_LEVEL)
print(f"{len(filtered_by_level)} / {len(df)} rows with Level >= {MIN_LEVEL!r}")
filtered_by_level.head()

In [ ]:
# Filter: time window [START, END], inclusive, UTC. None on either side means unbounded.
def filter_by_time(
    data: pd.DataFrame, start: str | None = None, end: str | None = None
) -> pd.DataFrame:
    '''Return rows with Timestamp in [start, end] (UTC, inclusive); None is unbounded.'''
    mask = pd.Series(True, index=data.index)
    if start is not None:
        mask &= data["Timestamp"] >= pd.Timestamp(start, tz="UTC")
    if end is not None:
        mask &= data["Timestamp"] <= pd.Timestamp(end, tz="UTC")
    return data[mask].copy()


START, END = None, None  # e.g. "2016-09-20T00:00:00", "2016-09-21T00:00:00"
filtered_by_time = filter_by_time(df, START, END)
print(f"{len(filtered_by_time)} / {len(df)} rows in the selected time window")
filtered_by_time.head()

In [ ]:
# Filter: Computer, exact match (case-sensitive, as Hayabusa writes it).
def filter_by_computer(data: pd.DataFrame, computer: str) -> pd.DataFrame:
    '''Return rows where Computer equals `computer` exactly.'''
    return data[data["Computer"] == computer].copy()


COMPUTER = df["Computer"].iloc[0] if not df.empty else ""
filtered_by_computer = filter_by_computer(df, COMPUTER)
print(f"{len(filtered_by_computer)} / {len(df)} rows for Computer == {COMPUTER!r}")
filtered_by_computer.head()

In [ ]:
# Filter: keyword (case-insensitive substring) across the free-text columns.
FREE_TEXT_COLUMNS: tuple[str, ...] = ("RuleTitle", "Details", "ExtraFieldInfo")


def filter_by_keyword(
    data: pd.DataFrame, keyword: str, columns: tuple[str, ...] = FREE_TEXT_COLUMNS
) -> pd.DataFrame:
    '''Return rows where `keyword` (case-insensitive) appears in any free-text column.'''
    mask = pd.Series(False, index=data.index)
    for col in columns:
        mask |= data[col].str.contains(keyword, case=False, na=False, regex=False)
    return data[mask].copy()


KEYWORD = "service"
filtered_by_keyword = filter_by_keyword(df, KEYWORD)
print(f"{len(filtered_by_keyword)} / {len(df)} rows matching keyword {KEYWORD!r}")
filtered_by_keyword.head()

## Visualisation

`display_timeline` (grouped by `Computer` or `Channel`) is called directly — no
`msticpyconfig.yaml` needed, no warning emitted. Any failure (offline environment, missing
optional dependency, bokeh version mismatch) is caught and degrades to the always-present
matplotlib histogram below, rather than halting the notebook.

In [ ]:
GROUP_BY = "Computer"  # or "Channel"

try:
    from msticpy.vis.timeline import display_timeline

    timeline_plot = display_timeline(
        data=df,
        time_column="Timestamp",
        group_by=GROUP_BY,
        title=f"Event timeline grouped by {GROUP_BY}",
        hide=True,
    )
    print("msticpy display_timeline rendered OK")
except Exception as exc:  # noqa: BLE001 -- deliberately broad: any msticpy/bokeh failure
    # must degrade to the matplotlib fallback below instead of halting the notebook.
    timeline_plot = None
    print(f"msticpy timeline unavailable, continuing with matplotlib fallback only: {exc}")

timeline_plot

In [ ]:
# Always-present matplotlib fallback: histogram of events over time. Independent of the
# msticpy cell above so it renders even when msticpy/bokeh is unavailable.
fig, ax = plt.subplots(figsize=(10, 4))
df["Timestamp"].dt.floor("h").value_counts().sort_index().plot(kind="bar", ax=ax, width=0.9)
ax.set_title("Event count over time (hourly, UTC)")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Event count")
n_ticks = len(ax.get_xticks())
if n_ticks > 20:
    for i, label in enumerate(ax.get_xticklabels()):
        label.set_visible(i % (n_ticks // 20 + 1) == 0)
plt.tight_layout()
plt.show()

## IOC extraction

Regex-only extraction of IPv4 addresses, domains, and MD5/SHA1/SHA256-length hex strings
from the free-text columns (`RuleTitle`, `Details`, `ExtraFieldInfo` — all three are 100%
non-null in Hayabusa's dfir-timeline output). Deduplicated into a `type` / `value` / `count`
table, using the same type labels (`ip`/`domain`/`hash`) that `lookup/soc-lookup`'s
`detect_type()` produces, so this table can be piped straight into it.

**No network or API call is made from this notebook.**

Windows command lines and .NET dotted identifiers in `Details`/`ExtraFieldInfo`
(`powershell.exe`, `RPCRT4.dll`, `g4g34pot.cmdline`, `System.Diagnostics.Process`, ...)
match the same dotted-label shape as a domain. A small denylist of common executable/
library/temp-file extensions filters out the most frequent false-positive class; it is a
noise reducer, not a TLD validator, and the resulting table is meant to be reviewed before
piping into `soc-lookup`.

In [ ]:
IPV4_RE = re.compile(
    r"\b(?:(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\b"
)
DOMAIN_RE = re.compile(
    r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,63}\b"
)
HASH_RE = re.compile(r"\b[a-fA-F0-9]{64}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{32}\b")

# Free-text columns are full of Windows command lines and .NET dotted identifiers
# (powershell.exe, RPCRT4.dll, g4g34pot.cmdline, System.Diagnostics.Process, ...) that
# match DOMAIN_RE's dotted-label shape but are not domains. This is a denylist of the
# most common false-positive last-labels seen in Hayabusa Details/ExtraFieldInfo text
# (executables, libraries, temp/compiler artifacts) -- a heuristic noise reducer, NOT a
# public-suffix/TLD validator (no such list ships in requirements.txt). Genuine .NET-style
# namespace tokens without a recognised extension (e.g. System.Diagnostics.Process) can
# still slip through; this table is meant to be reviewed before feeding into soc-lookup.
DOMAIN_FALSE_POSITIVE_EXTENSIONS: frozenset[str] = frozenset(
    {
        "exe", "dll", "sys", "ocx", "cpl", "scr", "drv",
        "cmdline", "tmp", "log", "bak", "dat", "pdb", "manifest",
        "bat", "cmd", "ps1", "psm1", "vbs", "js", "wsf",
        "net",  # Microsoft.NET path token dominates this dataset; also a real gTLD --
                # excluding it trades away genuine .net domains for far more common noise.
    }
)


def extract_iocs(data: pd.DataFrame, columns: tuple[str, ...] = FREE_TEXT_COLUMNS) -> pd.DataFrame:
    '''Extract IPv4 / domain / hash observables from free-text columns via regex only.

    No network or API call is made here -- lookup/soc-lookup performs the actual
    enrichment. Returns a deduplicated table with columns type/value/count, using the
    same type labels (ip/domain/hash) as soc-lookup's own detect_type().
    '''
    text = "\n".join(
        data[list(columns)].fillna("").astype(str).agg(" ".join, axis=1).tolist()
    )

    ipv4_hits = IPV4_RE.findall(text)
    hash_hits = [h.lower() for h in HASH_RE.findall(text)]
    ipv4_set = set(ipv4_hits)
    # Domain regex also matches dotted IPv4 addresses -- exclude those explicitly, plus
    # the common Windows filename/extension false positives above.
    domain_hits = [
        d
        for d in DOMAIN_RE.findall(text)
        if d not in ipv4_set
        and d.rsplit(".", 1)[-1].lower() not in DOMAIN_FALSE_POSITIVE_EXTENSIONS
    ]

    hits: list[tuple[str, str]] = (
        [("ip", v) for v in ipv4_hits]
        + [("domain", v) for v in domain_hits]
        + [("hash", v) for v in hash_hits]
    )

    if not hits:
        return pd.DataFrame(columns=["type", "value", "count"])

    hits_df = pd.DataFrame(hits, columns=["type", "value"])
    ioc_table = hits_df.groupby(["type", "value"]).size().reset_index(name="count")
    return ioc_table.sort_values(["type", "count"], ascending=[True, False]).reset_index(drop=True)


ioc_table = extract_iocs(df)
print(f"Extracted {len(ioc_table)} unique IOC(s)")
ioc_table

## Export

Writes a markdown summary (time range, counts, top findings, extracted IOCs) next to the
input CSV, continuing `evtx-triage.sh`'s `NNN-` output numbering (e.g.
`005-timeline-summary.md` after `001`-`004`) rather than a hardcoded number.

In [ ]:
def next_output_number(directory: Path) -> int:
    '''Return the next sequential 3-digit output number for `directory`.

    Scans existing NNN-prefixed entries written by evtx-triage.sh (001-hayabusa-timeline.csv,
    002-*, 003-*, 004-*) and returns max + 1, so this notebook's export continues that
    numbering instead of hardcoding a fixed number.
    '''
    pattern = re.compile(r"^(\d{3})-")
    numbers = [
        int(m.group(1))
        for p in directory.iterdir()
        if (m := pattern.match(p.name))
    ]
    return max(numbers, default=0) + 1


output_dir = CSV_PATH.parent
summary_number = next_output_number(output_dir)
summary_path = output_dir / f"{summary_number:03d}-timeline-summary.md"
summary_path

In [ ]:
top_rule_titles = df["RuleTitle"].value_counts().head(10)

summary_lines: list[str] = [
    f"# Timeline Summary --- {CSV_PATH.name}",
    "",
    f"Generated: {pd.Timestamp.now(tz='UTC').isoformat()}",
    "",
    "## Time range",
    "",
    f"- Start: {time_min.isoformat()}",
    f"- End: {time_max.isoformat()}",
    f"- Total events: {len(df)}",
    "",
    "## Counts by Level",
    "",
    "| Level | Count |",
    "|---|---|",
]
summary_lines += [f"| {level} | {count} |" for level, count in level_counts.items()]

summary_lines += ["", "## Counts by Channel", "", "| Channel | Count |", "|---|---|"]
summary_lines += [f"| {channel} | {count} |" for channel, count in channel_counts.items()]

summary_lines += ["", "## Counts by Computer", "", "| Computer | Count |", "|---|---|"]
summary_lines += [f"| {computer} | {count} |" for computer, count in computer_counts.items()]

summary_lines += ["", "## Top 10 findings (RuleTitle)", "", "| RuleTitle | Count |", "|---|---|"]
summary_lines += [f"| {title} | {count} |" for title, count in top_rule_titles.items()]

summary_lines += ["", "## Extracted IOCs", "", f"Total unique IOCs: {len(ioc_table)}", ""]
if not ioc_table.empty:
    summary_lines += ["| Type | Value | Count |", "|---|---|---|"]
    summary_lines += [
        f"| {row['type']} | {row['value']} | {row['count']} |"
        for _, row in ioc_table.iterrows()
    ]

summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print(f"Wrote summary to {summary_path}")